<a href="https://colab.research.google.com/github/mnsbharadwaj/AI-NLP/blob/master/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**RAG : Retrieval-Augmented Generation**

RAG (Retrieval-Augmented Generation) is a powerful method that combines retrieval-based and generation-based techniques for answering questions or generating text. The core idea behind RAG is to retrieve relevant documents (contexts) and then use them to generate accurate, grounded, and fluent responses.

**Core Components of RAG:**

RAG consists of two major components:


**Retriever (Dense Retriever)** : Retrieves top-k relevant documents using embeddings.

**Generator** (e.g., BART, T5):	Generates final answer based on retrieved documents.


**Retriever (Dense Passage Retriever - DPR)**

  Purpose:
Convert input queries and documents into dense vectors (embeddings) and retrieve the top-k similar documents.

Key Modules:

Context Encoder: Embeds documents.

Question Encoder: Embeds query.

Similarity Search: Typically via FAISS.

In [1]:
!pip install faiss-cpu
!pip install chromadb

In [2]:
from transformers import DPRQuestionEncoder, DPRContextEncoder, DPRQuestionEncoderTokenizer, DPRContextEncoderTokenizer
import torch
import faiss

# Load pretrained encoders and tokenizers
question_encoder = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
question_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-question_encoder-single-nq-base")

context_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
context_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")

# Sample corpus
documents = [
    "The capital of France is Paris.",
    "Hugging Face provides transformers for NLP.",
    "The Moon orbits the Earth.",
]

# Encode documents
def encode_documents(docs):
    inputs = context_tokenizer(docs, return_tensors='pt', padding=True, truncation=True)
    with torch.no_grad():
        embeddings = context_encoder(**inputs).pooler_output
    return embeddings

doc_embeddings = encode_documents(documents)

# FAISS index
index = faiss.IndexFlatL2(doc_embeddings.size(1))  # dimension
index.add(doc_embeddings.numpy())

# Encode query
query = "What is the capital of France?"
query_inputs = question_tokenizer(query, return_tensors='pt')
with torch.no_grad():
    query_embedding = question_encoder(**query_inputs).pooler_output

# Retrieve top-k (e.g., 2) documents
top_k = 2
distances, indices = index.search(query_embedding.numpy(), top_k)

# Show results
print("Query:", query)
for idx in indices[0]:
    print("Retrieved:", documents[idx])


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of the model checkpoint at facebook/dpr-question_encoder-single-nq-base were not used when initializing DPRQuestionEncoder: ['question_encoder.bert_model.pooler.dense.bias', 'question_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRQuestionEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This 

Query: What is the capital of France?
Retrieved: The capital of France is Paris.
Retrieved: The Moon orbits the Earth.


## **Generator (e.g., BART, T5)**
Purpose:

Generates an answer conditioned on the retrieved documents.

How It Works:

Takes [query + retrieved_document] as input.

Generates an answer using a seq2seq transformer (e.g., facebook/bart-large or google/flan-t5).

In [3]:
from transformers import BartTokenizer, BartForConditionalGeneration

# Load BART generator
gen_tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')
generator = BartForConditionalGeneration.from_pretrained('facebook/bart-large')

# Combine query and top retrieved document
retrieved_context = documents[indices[0][0]]
input_text = f"question: {query} context: {retrieved_context}"
inputs = gen_tokenizer(input_text, return_tensors='pt', truncation=True, padding=True)

# Generate answer
with torch.no_grad():
    output_ids = generator.generate(**inputs, max_length=50)

print("Generated Answer:", gen_tokenizer.decode(output_ids[0], skip_special_tokens=True))


Generated Answer: question: What is the capital of France? context: The president of France is Paris.


## **Code Generation**

In [4]:
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# Load your embedding model and generator
embedder = SentenceTransformer('microsoft/codebert-base')
generator = pipeline("text2text-generation", model="Salesforce/codet5-base")

# Step 1: Your code knowledge base
code_snippets = [
    "def connect_to_db(uri):\n    return sqlite3.connect(uri)",
    "def add(a, b):\n    return a + b"
]

# Step 2: Embed and store
embeddings = embedder.encode(code_snippets)
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings))

# Step 3: User prompt (user query)
user_query = "How do I connect to a PostgreSQL database in Python?"

# Step 4: Embed query and retrieve top-k
query_vec = embedder.encode([user_query])
_, indices = index.search(np.array(query_vec), 2)
retrieved_context = "\n\n".join([code_snippets[i] for i in indices[0]])

# ✅ Step 5: BUILD PROMPT (this is what you're missing!)
prompt = f"""
You are a coding assistant. Use the following code snippets as reference:

{retrieved_context}

Now, {user_query}
"""

# Step 6: Generate answer
result = generator(prompt, max_length=200, do_sample=False)
print("🧠 Generated Code:\n", result[0]['generated_text'])


Device set to use cpu
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧠 Generated Code:
 sqlite3.create_connection(uri)

Now, How do Ia PostgreSQL database?sqlite3.create_connection(uri)

Now, How do ISQLite3.create_connection(uri)

Now, How do ISQLite3.create_connection(uri)

Now, How do ISQLite3.create_connection(uri)

Now, How do ISQLite3.
